In [1]:
import gc, os
import json
import time
import optuna
import pyarrow

import polars as pl
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from catboost import CatBoostClassifier, Pool
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt

/home/d.golomolzin/kiber-polka/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_path = 'data/train/train_main_features.parquet'
target_path = 'data/train/train_target.parquet'

def init_data_start(target_id: str, train, target):
    # Этап 1: загрузка данных
    target_by_id = target.select(target_id).to_series()

    # Этап 2: разделяем на valid/train
    X_train, X_valid, y_train, y_valid = train_test_split(
        train.to_pandas(),
        target_by_id.to_numpy(), 
        test_size=0.2, 
        random_state=42
    )

    X_train = X_train.reset_index(drop=True)
    X_valid = X_valid.reset_index(drop=True)

    # Этап 4: выбираем cat_feature с ними работает catboost
    cat_feature_names = [col for col in X_train.columns if col.startswith("cat_feature")]
    X_train[cat_feature_names] = X_train[cat_feature_names].astype(str)
    X_valid[cat_feature_names] = X_valid[cat_feature_names].astype(str)

    # Этап 5: для catboost pool лучше
    train_pool = Pool(X_train, y_train, cat_feature_names)
    valid_pool = Pool(X_valid, y_valid, cat_feature_names)

    return train_pool, valid_pool, X_train, y_train

In [3]:
def objective(trial: optuna.Trial, train: Pool, valid: Pool):
    params = {
        'eval_metric': 'AUC',
        'verbose': 50, # отчёт каждые N новых деревьев
        'use_best_model': True,
        'early_stopping_rounds': 50, # если метрика не растёт 50 деревьев, пруним
        'task_type': 'GPU',
        'thread_count': 20,  # кол-во используемых потоков(-1 макс)
        'iterations': 1_200, # специально много, потому что ставка не на это
        'depth': trial.suggest_int('depth', 4, 7),
        'learning_rate': trial.suggest_float('learning_rate', .01, .3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1., 50., log=True),
        #'subsample': trial.suggest_float('subsample', .5, 1.),
        #'colsample_bylevel': trial.suggest_float('colsample_bylevel', .5, 1.),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'random_strength': trial.suggest_int('random_strength', 1, 7),
    }

    model = CatBoostClassifier(**params)

    model.fit(train, eval_set=valid)

    best_score = model.get_best_score()['validation']['AUC']

    # ДЛЯ ПРУНИНГА
    # длина AUC-списка = кол-ву деревьев (iterations) ЛИБО пока не обрубится
    scores = model.get_evals_result()['validation']['AUC']
    for epoch, score in enumerate(scores, 1):
        trial.report(score, epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()
    
    # ДЛЯ OPTUNA
    # потом можно переделать чтобы рисовать кривые обучения
    return best_score

In [4]:
def get_bst_par(train: Pool, valid: Pool):
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=1, # кол-во честных попыток (без прнунинга)
            n_warmup_steps=20
        )
    )
    study.optimize(
        lambda trial: objective(trial, train, valid),
        n_trials=2,    # ограничение в 30попыток поиска параметров
        gc_after_trial=True,
    )
    
    return study.best_params

In [5]:
def get_bst_features(best_params, train: Pool, valid: Pool):
    model = CatBoostClassifier(
        **best_params,
        task_type='GPU',
        verbose=False
    )
    model.fit(train, eval_set=valid)
    
    # нужно отфильтровать и оставить top-N признаков
    feature_importance = model.get_feature_importance(prettified=True)

    # НУЖНО ОПРЕДЕЛИТЬ ОПТИМАЛЬНОЕ КОЛ-ВО ВАЖНЫХ ПРИЗНАКОВ
    N = 150
    best_features = feature_importance.iloc[:, 0].head(N).tolist()

    return best_features

In [6]:
def oof_one_target(best_features, best_params, X_train, y_train):
    X_train = X_train[best_features]
    category = [col for col in X_train.columns if col.startswith('cat_feature')]

    kf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

    # samples * 1 <- вектор предсказаний для каждой строчки (1, потому что мы обучаем тут одну определенную модель)
    # потом простэкуем их все и получим samples * 41, что и будет являться обучащими данными для второй модели
    preds_model = np.zeros(len(X_train))

    for train_idx, valid_idx in kf.split(X_train, y_train):

        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train[train_idx], y_train[valid_idx]

        train_fold = Pool(data=X_tr, label=y_tr,cat_features=category)
        valid_fold = Pool(data=X_val, label=y_val,cat_features=category)

        model = CatBoostClassifier(
            **best_params,
            task_type='GPU',
            verbose=False
        )
        model.fit(train_fold, eval_set=valid_fold)
        preds_model[valid_idx] = model.predict_proba(valid_fold)[:,1]

        del model, train_fold, valid_fold
        gc.collect()

    return preds_model, category 

In [7]:
def save_snapshot(best_params, best_features, target_idx, X_train, y_train, category):
    X_train = X_train[best_features]
    model = CatBoostClassifier(
        **best_params,
        task_type='GPU',
        verbose=False
    )
    
    X_all_train = Pool(data=X_train, label=y_train, cat_features=category)
    model.fit(X_all_train)

    Path('snapshots').mkdir(exist_ok=True)
    Path('configs').mkdir(exist_ok=True)

    model.save_model(f'snapshots/{target_idx}.cbm')

    data = {
        'target': target_idx,
        'best_params': best_params,
        'best_features': best_features,
        'category': category
    }
    
    with open(f'configs/{target_idx}.json', 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4)

In [8]:
def save_oof_column(target_idx: str, preds: np.ndarray, filepath: str = 'meta_data.parquet'):
    if os.path.exists(filepath):
        # Читаем существующий файл и добавляем колонку
        meta_df = pl.read_parquet(filepath)
        meta_df = meta_df.with_columns(pl.Series(target_idx, preds))
    else:
        # Создаём новый DataFrame с первой колонкой
        meta_df = pl.DataFrame({target_idx: preds})
    
    meta_df.write_parquet(filepath)
    print(f"✅ Сохранён столбец '{target_idx}' в {filepath}")


In [9]:
train = pl.read_parquet(train_path).drop('customer_id')
target = pl.read_parquet(target_path).drop('customer_id')
columns_tar = target.columns

oof_predictions = {} # словарь для предсказаний
idx_col = [0, 1] # таргеты которые хочешь обработать за сессию


In [10]:
target_idx = columns_tar[1]

print(f"TARGET: {target_idx}")
time_target = time.time()
print('Этап 1: получаем data')
train_pool, valid_pool, X_train, y_train = init_data_start(target_idx, train, target)

TARGET: target_1_2
Этап 1: получаем data


In [11]:
print('Этап 2: получаем лучшие гиперпараметры таргета')
best_params = get_bst_par(train_pool, valid_pool)

[I 2026-03-06 17:12:33,637] A new study created in memory with name: no-name-6615bbea-44a8-4fca-80c5-6c8660d40f6f


Этап 2: получаем лучшие гиперпараметры таргета


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.5909036	best: 0.5909036 (0)	total: 153ms	remaining: 3m 3s
50:	test: 0.7563055	best: 0.7563055 (50)	total: 2.46s	remaining: 55.5s
100:	test: 0.7622744	best: 0.7622744 (100)	total: 4.74s	remaining: 51.6s
150:	test: 0.7653687	best: 0.7655228 (147)	total: 7s	remaining: 48.6s
200:	test: 0.7733329	best: 0.7733329 (200)	total: 9.21s	remaining: 45.8s
250:	test: 0.7749189	best: 0.7749528 (248)	total: 11.4s	remaining: 43.1s
300:	test: 0.7791646	best: 0.7791646 (300)	total: 13.6s	remaining: 40.8s
350:	test: 0.7828103	best: 0.7828103 (350)	total: 16s	remaining: 38.6s
400:	test: 0.7864196	best: 0.7864196 (400)	total: 18.2s	remaining: 36.3s
450:	test: 0.7874063	best: 0.7875228 (448)	total: 20.5s	remaining: 34s
500:	test: 0.7885619	best: 0.7890023 (494)	total: 22.8s	remaining: 31.8s
550:	test: 0.7916380	best: 0.7916380 (550)	total: 25.1s	remaining: 29.5s
600:	test: 0.7938864	best: 0.7941438 (594)	total: 27.4s	remaining: 27.3s
650:	test: 0.7947310	best: 0.7947310 (650)	total: 29.7s	remainin

[I 2026-03-06 17:13:20,626] Trial 0 finished with value: 0.8009783923625946 and parameters: {'depth': 5, 'learning_rate': 0.2536999076681772, 'l2_leaf_reg': 17.524101118128137, 'min_data_in_leaf': 30, 'random_strength': 2}. Best is trial 0 with value: 0.8009783923625946.
Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.5909036	best: 0.5909036 (0)	total: 37.9ms	remaining: 45.4s
50:	test: 0.7016819	best: 0.7016819 (50)	total: 1.89s	remaining: 42.5s
100:	test: 0.7179086	best: 0.7179086 (100)	total: 3.74s	remaining: 40.7s
150:	test: 0.7273236	best: 0.7273236 (150)	total: 5.61s	remaining: 39s
200:	test: 0.7322632	best: 0.7325434 (193)	total: 7.46s	remaining: 37.1s
250:	test: 0.7359578	best: 0.7359578 (250)	total: 9.35s	remaining: 35.4s
300:	test: 0.7445465	best: 0.7445465 (300)	total: 11.2s	remaining: 33.4s
350:	test: 0.7499244	best: 0.7499244 (350)	total: 13s	remaining: 31.5s
400:	test: 0.7539321	best: 0.7539321 (400)	total: 14.9s	remaining: 29.7s
450:	test: 0.7565141	best: 0.7565141 (450)	total: 16.8s	remaining: 27.8s
500:	test: 0.7585919	best: 0.7586465 (494)	total: 18.6s	remaining: 25.9s
550:	test: 0.7613115	best: 0.7613115 (550)	total: 20.5s	remaining: 24.1s
600:	test: 0.7629388	best: 0.7629396 (599)	total: 22.4s	remaining: 22.3s
650:	test: 0.7646351	best: 0.7646351 (649)	total: 24.2s	rema

[I 2026-03-06 17:14:09,825] Trial 1 pruned. 


In [12]:
print('Этап 3: получаем лучшие фичи в датасете для таргета')
best_features = get_bst_features(best_params, train_pool, valid_pool)

Этап 3: получаем лучшие фичи в датасете для таргета


In [13]:
print('Этап 4: получаем столбец предсказаний по target_id')
preds_target, category = oof_one_target(best_features, best_params, X_train, y_train)

Этап 4: получаем столбец предсказаний по target_id


In [14]:
print('Этап 5: добавляем результат предсказания по таргету в общий список')
oof_predictions[target_idx] = preds_target

Этап 5: добавляем результат предсказания по таргету в общий список


In [15]:
print('Этап 6: сохраняем данные о модели')
save_snapshot(best_params, best_features, target_idx, X_train, y_train, category)

Этап 6: сохраняем данные о модели


In [16]:
print('Этап 7: сохраняем preds_target для каждого таргета')
save_oof_column(target_idx, preds_target, 'meta_data.parquet')

Этап 7: сохраняем preds_target для каждого таргета
✅ Сохранён столбец 'target_1_2' в meta_data.parquet


In [17]:
# time_target = time.time() - time_target
# print(f'Work time: {time_target}\n')

In [18]:
a = pl.read_parquet(r'/home/d.golomolzin/kiber-polka/meta_data.parquet')

In [20]:
a

target_1_1,target_1_2
f64,f64
0.004554,0.002074
0.001229,0.004768
0.011658,0.021629
0.001072,0.002484
0.002424,0.004518
…,…
0.03613,0.003814
0.001247,0.001299
0.000963,0.001403
